# 01 — Data Exploration
**Lane:** Refresh / Content Opportunity Scoring

> NOTE: `load_search_data()` currently returns synthetic data shaped like the real
> FlyRank warehouse. See `_lib.py` docstring for the exact swap-in point
> (the DuckDB + `hf://` query from starter notebook 03). Every notebook downstream
> of this one is written against this schema, so swapping the data source is
> the only change needed to go from demo to real.

In [1]:
import sys
sys.path.insert(0, "..")
from _lib import load_search_data
import pandas as pd

df = load_search_data()
print(df.shape)
df.head()

(10000, 9)


,page_id,week,clicks,impressions,ctr,position,word_count,category,days_since_publish
0,page_0000,0,129,2687,0.0480,6.5,2260,blog,1225
1,page_0000,1,142,2811,0.0505,6.5,2260,blog,1232
2,page_0000,2,141,2924,0.0482,6.3,2260,blog,1239
3,page_0000,3,137,3207,0.0427,10.4,2260,blog,1246
4,page_0000,4,148,2587,0.0572,7.2,2260,blog,1253


In [2]:
# Basic shape / grain checks
print("Unique pages:", df['page_id'].nunique())
print("Weeks covered:", df['week'].min(), "-", df['week'].max())
print("Rows per page (should be constant = n_weeks):")
print(df.groupby('page_id').size().describe())

Unique pages: 500
Weeks covered: 0 - 19
Rows per page (should be constant = n_weeks):
count    500.0
mean      20.0
std        0.0
min       20.0
25%       20.0
50%       20.0
75%       20.0
max       20.0
dtype: float64


In [3]:
# Sanity check distributions
df[['clicks','impressions','ctr','position']].describe()

,clicks,impressions,ctr,position
count,10000.000000,10000.000000,10000.000000,10000.000000
mean,68.886300,2107.767900,0.031941,7.034340
std,60.235546,1337.620757,0.015418,3.063024
min,0.000000,20.000000,0.000000,1.000000
25%,21.000000,951.750000,0.020900,4.900000
50%,53.000000,1965.000000,0.032100,7.000000
75%,101.000000,3073.500000,0.042100,9.100000
max,389.000000,7587.000000,0.100000,19.300000


In [4]:
# Category mix
df.groupby('category')['page_id'].nunique().sort_values(ascending=False)

category
faq           94
comparison    88
guide         81
landing       80
blog          79
product       78
Name: page_id, dtype: int64

## Notes
- Grain is page x week — one row per page per week, consistent with expected schema.
- `ctr` and `position` look reasonable (ctr mostly under ~10%, position 1-100).
- Real data swap-in: replace `load_search_data()` with the DuckDB `hf://` query,
  keeping columns: `page_id, week (or date), clicks, impressions, ctr, position,
  word_count, category, days_since_publish`.